# Detector Benchmark and Model Selection

Reproducible experiment record for AeroXAI anomaly detection.

This notebook replaces the orchestration in:
- `ml/detection/run_baseline.py`
- `ml/detection/benchmark.py`

Reusable detection logic remains under `ml/detection/`.

The benchmark protocol is fixed:
- fit models on the training partition only;
- calibrate thresholds on the calibration partition only;
- use the test partition only for final evaluation;
- apply the same EWMA, persistence, alert-episode extraction, and evaluation logic to every detector.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()

if not (ROOT / "ml").exists() and (ROOT.parent / "ml").exists():
    ROOT = ROOT.parent

if not (ROOT / "ml").exists():
    raise RuntimeError(
        "Could not locate repository root. "
        "Open this notebook from the xAI-Compressor repository."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Repository root: {ROOT}")

Repository root: C:\Users\Quoc Thai\Downloads\AI\xAI-Compressor


In [2]:
import json
from typing import Any

import pandas as pd
import yaml

from ml.data.features import model_feature_columns
from ml.detection.alerts import (
    causal_ewma,
    extract_alert_episodes,
    persistent_alerts,
)
from ml.detection.evaluate import evaluate_detection
from ml.detection.isolation_forest import (
    fit_isolation_forest,
    score_isolation_forest,
)
from ml.detection.pca_detector import (
    fit_pca_detector,
    score_pca_detector,
)
from ml.detection.robust_z import (
    fit_robust_z,
    score_robust_z,
)

METROPT_CONFIG_PATH = ROOT / "configs" / "metropt.yaml"
DETECTION_CONFIG_PATH = ROOT / "configs" / "detection.yaml"
OUTPUT_PATH = ROOT / "docs" / "detector_benchmark.json"

with METROPT_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    metropt = yaml.safe_load(handle)

with DETECTION_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    detection = yaml.safe_load(handle)

preprocessing = metropt["preprocessing"]
detector_config = detection["detector"]
alert_config = detection["alerting"]
evaluation_config = detection["evaluation"]
pca_config = detection["pca"]
selection_config = detection["selection"]
incidents = metropt["dataset"]["reported_incidents"]

## Load frozen partitions

In [3]:
def load_partition(path: Path) -> pd.DataFrame:
    data = pd.read_parquet(path)
    data.index = pd.DatetimeIndex(
        data.index,
        name="timestamp",
    )
    return data.sort_index()

train = load_partition(
    ROOT / preprocessing["train_file"]
)

calibration = load_partition(
    ROOT / preprocessing["calibration_file"]
)

test = load_partition(
    ROOT / preprocessing["test_file"]
)

feature_columns = model_feature_columns(
    train.columns
)

pd.DataFrame(
    {
        "rows": [
            len(train),
            len(calibration),
            len(test),
        ],
        "start": [
            train.index.min(),
            calibration.index.min(),
            test.index.min(),
        ],
        "end": [
            train.index.max(),
            calibration.index.max(),
            test.index.max(),
        ],
    },
    index=["train", "calibration", "test"],
)

,rows,start,end
train,5271,2020-02-01 00:05:00,2020-02-21 23:55:00
calibration,1796,2020-02-22 01:00:00,2020-02-28 23:55:00
test,43221,2020-03-01 04:05:00,2020-09-01 03:55:00


## Shared thresholding and operational evaluation

In [4]:
def evaluate_scores(
    *,
    calibration_scores: pd.Series,
    test_scores: pd.Series,
) -> tuple[dict[str, Any], float]:
    calibration_smoothed = causal_ewma(
        calibration_scores,
        alpha=float(
            alert_config["ewma_alpha"]
        ),
        reset_gap_minutes=int(
            alert_config[
                "reset_gap_minutes"
            ]
        ),
    )

    threshold = float(
        calibration_smoothed.quantile(
            float(
                alert_config[
                    "threshold_quantile"
                ]
            ),
            interpolation="higher",
        )
    )

    test_smoothed = causal_ewma(
        test_scores,
        alpha=float(
            alert_config["ewma_alpha"]
        ),
        reset_gap_minutes=int(
            alert_config[
                "reset_gap_minutes"
            ]
        ),
    )

    _, alerts = persistent_alerts(
        test_smoothed,
        threshold=threshold,
        required_hits=int(
            alert_config[
                "persistence_hits"
            ]
        ),
        window_bins=int(
            alert_config[
                "persistence_window"
            ]
        ),
        reset_gap_minutes=int(
            alert_config[
                "reset_gap_minutes"
            ]
        ),
    )

    episodes = extract_alert_episodes(
        alerts,
        merge_minutes=int(
            alert_config[
                "merge_minutes"
            ]
        ),
        reset_gap_minutes=int(
            alert_config[
                "reset_gap_minutes"
            ]
        ),
    )

    metrics = evaluate_detection(
        scores=test_smoothed,
        alerts=alerts,
        episodes=episodes,
        incidents=incidents,
        early_warning_hours=int(
            evaluation_config[
                "early_warning_hours"
            ]
        ),
        late_tolerance_hours=int(
            evaluation_config[
                "late_tolerance_hours"
            ]
        ),
        bin_minutes=int(
            evaluation_config[
                "bin_minutes"
            ]
        ),
    )

    return metrics, threshold

## Robust-Z baseline

In [5]:
robust_model = fit_robust_z(
    train,
    feature_columns,
    top_k=int(
        detector_config["top_k"]
    ),
    min_scale=float(
        detector_config["min_scale"]
    ),
)

robust_calibration_scores, _ = score_robust_z(
    calibration,
    robust_model,
)

robust_test_scores, _ = score_robust_z(
    test,
    robust_model,
)

robust_metrics, robust_threshold = evaluate_scores(
    calibration_scores=robust_calibration_scores,
    test_scores=robust_test_scores,
)

robust_metrics["model"] = {
    "name": "robust_z_topk",
    "input_feature_count":
        len(feature_columns),
    "active_feature_count":
        len(robust_model.features),
    "dropped_feature_count":
        len(robust_model.dropped_features),
    "dropped_features":
        robust_model.dropped_features,
    "top_k":
        robust_model.top_k,
    "threshold":
        robust_threshold,
}

## PCA reconstruction detector

In [6]:
variance_retained = float(
    pca_config["variance_retained"]
)

pca_detector = fit_pca_detector(
    train,
    feature_columns,
    variance_retained=variance_retained,
)

pca_calibration_scores = score_pca_detector(
    calibration,
    pca_detector,
)

pca_test_scores = score_pca_detector(
    test,
    pca_detector,
)

pca_metrics, pca_threshold = evaluate_scores(
    calibration_scores=pca_calibration_scores,
    test_scores=pca_test_scores,
)

pca_metrics["model"] = {
    "name": "pca_reconstruction",
    "input_feature_count":
        len(feature_columns),
    "variance_retained":
        variance_retained,
    "components":
        int(pca_detector.model.n_components_),
    "explained_variance_ratio":
        float(
            pca_detector.model
            .explained_variance_ratio_
            .sum()
        ),
    "threshold":
        pca_threshold,
}

## Isolation Forest comparison

In [7]:
n_estimators = 300
random_state = 42

if_detector = fit_isolation_forest(
    train,
    feature_columns,
    n_estimators=n_estimators,
    random_state=random_state,
)

if_calibration_scores = score_isolation_forest(
    calibration,
    if_detector,
)

if_test_scores = score_isolation_forest(
    test,
    if_detector,
)

if_metrics, if_threshold = evaluate_scores(
    calibration_scores=if_calibration_scores,
    test_scores=if_test_scores,
)

if_metrics["model"] = {
    "name": "isolation_forest",
    "input_feature_count":
        len(feature_columns),
    "n_estimators":
        n_estimators,
    "random_state":
        random_state,
    "threshold":
        if_threshold,
}

## Comparison table

In [8]:
def compact_summary(
    metrics: dict[str, Any],
) -> dict[str, Any]:
    keys = [
        "timely_incident_recall",
        "anytime_incident_recall",
        "pre_onset_incident_recall",
        "incident_overlap_recall",
        "false_alerts_per_24h",
        "time_in_alert_fraction",
        "relevant_episode_precision",
        "pr_auc",
        "episodes_total",
        "false_episodes",
    ]

    return {
        key: metrics[key]
        for key in keys
    }

summary = {
    "robust_z":
        compact_summary(robust_metrics),
    "pca":
        compact_summary(pca_metrics),
    "isolation_forest":
        compact_summary(if_metrics),
}

summary_frame = pd.DataFrame(summary).T
summary_frame

,timely_incident_recall,anytime_incident_recall,pre_onset_incident_recall,incident_overlap_recall,false_alerts_per_24h,time_in_alert_fraction,relevant_episode_precision,pr_auc,episodes_total,false_episodes
robust_z,1.0,1.0,1.0,1.0,2.771986,0.160848,0.025761,0.091323,427.0,416.0
pca,1.0,1.0,0.5,1.0,0.886236,0.075496,0.063380,0.249168,142.0,133.0
isolation_forest,1.0,1.0,1.0,1.0,4.997571,0.481757,0.032258,0.441400,775.0,750.0


## Frozen selection

`configs/detection.yaml` names the primary detector used by the next stage. The other detectors stay as benchmark evidence.

Current primary detector: **PCA reconstruction**.

In [9]:
print(
    "Configured primary detector:",
    selection_config["primary"],
)

pd.DataFrame(
    pca_metrics["incident_results"]
)

Configured primary detector: pca_reconstruction


,id,timely_detected,anytime_detected,first_alert,lead_hours,delay_hours
0,1,True,True,2020-04-18 00:40:00,NaN,0.666667
1,2,True,True,2020-05-29 21:20:00,2.166667,NaN
2,3,True,True,2020-06-05 10:10:00,NaN,0.166667
3,4,True,True,2020-07-15 08:45:00,5.750000,NaN


## Write benchmark report

In [10]:
report = {
    "evaluation_protocol": {
        "train_rows":
            len(train),
        "calibration_rows":
            len(calibration),
        "test_rows":
            len(test),
        "model_feature_count":
            len(feature_columns),
        "threshold_quantile":
            alert_config[
                "threshold_quantile"
            ],
        "ewma_alpha":
            alert_config[
                "ewma_alpha"
            ],
        "persistence_hits":
            alert_config[
                "persistence_hits"
            ],
        "persistence_window":
            alert_config[
                "persistence_window"
            ],
        "early_warning_hours":
            evaluation_config[
                "early_warning_hours"
            ],
        "late_tolerance_hours":
            evaluation_config[
                "late_tolerance_hours"
            ],
    },
    "summary": summary,
    "details": {
        "robust_z":
            robust_metrics,
        "pca":
            pca_metrics,
        "isolation_forest":
            if_metrics,
    },
}

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)
print(
    f"Benchmark written to: {OUTPUT_PATH}"
)

Benchmark written to: C:\Users\Quoc Thai\Downloads\AI\xAI-Compressor\docs\detector_benchmark.json
